In [1]:
import sqlite3
import pandas as pd

# 1.paytm_payments.db
conn = sqlite3.connect('paytm_payments.db')
cursor = conn.cursor()

# 2.safety net
cursor.executescript('''
    DROP TABLE IF EXISTS transactions;
    DROP TABLE IF EXISTS merchants;
    DROP TABLE IF EXISTS users;
''')

# 3. Schema for the database
cursor.executescript('''
    CREATE TABLE merchants (
        merchant_id INTEGER PRIMARY KEY,
        merchant_name TEXT,
        category TEXT,
        region TEXT
    );

    CREATE TABLE users (
        user_id INTEGER PRIMARY KEY,
        signup_date TIMESTAMP
    );

    CREATE TABLE transactions (
        transaction_id TEXT PRIMARY KEY,
        user_id INTEGER,
        merchant_id INTEGER,
        transaction_time TIMESTAMP,
        amount_inr INTEGER,
        payment_method TEXT,
        status TEXT,
        risk_score INTEGER,
        FOREIGN KEY(user_id) REFERENCES users(user_id),
        FOREIGN KEY(merchant_id) REFERENCES merchants(merchant_id)
    );
''')

# 4. Data from CSV files
merchants_df = pd.read_csv('merchants.csv')
users_df = pd.read_csv('users.csv')
ledger_df = pd.read_csv('ledger.csv')

# 5. SQL tables
merchants_df.to_sql('merchants', conn, if_exists='append', index=False)
users_df.to_sql('users', conn, if_exists='append', index=False)
ledger_df.to_sql('transactions', conn, if_exists='append', index=False)

print("paytm_payments.db successfully created, schema enforced, and data loaded!")

paytm_payments.db successfully created, schema enforced, and data loaded!


In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('paytm_payments.db')

query_1="""Select merchant_name
From merchants
where region='South'
"""
pd.read_sql(query_1, conn)

,merchant_name
0,Merchant_001
1,Merchant_005
2,Merchant_006
3,Merchant_022
4,Merchant_025
5,Merchant_028
6,Merchant_034
7,Merchant_037
8,Merchant_040


In [3]:
query_2="""
select distinct user_id
from transactions
where status='chargeback'
"""
query_2_1="""
select count (*) as total_chargebacks
from transactions
where status='chargeback'
"""

query_2_2="""
select sum(amount_inr) as total_chargeback_amount
from transactions
where status='chargeback'
"""

print("Total Count of Chargebacks")
display(pd.read_sql(query_2_1,conn))

print("Total Chargeback Amount")
display(pd.read_sql(query_2_2,conn))

print("Unique [user_ids] Affected- ")
display(pd.read_sql(query_2,conn))

Total Count of Chargebacks


,total_chargebacks
0,28


Total Chargeback Amount


,total_chargeback_amount
0,54472


Unique [user_ids] Affected- 


,user_id
0,338
1,245
2,211
3,280
4,328
5,297
6,114
7,226
8,168
9,314


In [4]:
query_3="""
select u.user_id, u.signup_date, t.transaction_time, t.status,
round(julianday(t.transaction_time)-julianday(u.signup_date),3) as time_before_transaction
from users u
join transactions t on u.user_id=t.user_id
where t.status='chargeback'
and julianday(t.transaction_time)-julianday(u.signup_date)>=0
and julianday(t.transaction_time)-julianday(u.signup_date)<30
"""

print("Burner Accounts (0-30 day old Txns)")
display(pd.read_sql(query_3,conn))

Burner Accounts (0-30 day old Txns)


,user_id,signup_date,transaction_time,status,time_before_transaction
0,351,2026-01-15 06:00:00,2026-01-30 06:00:00,chargeback,15.0
1,352,2025-12-31 12:00:00,2026-01-11 12:00:00,chargeback,11.0
2,353,2026-01-10 14:00:00,2026-01-21 14:00:00,chargeback,11.0
3,354,2025-12-29 19:00:00,2026-01-21 19:00:00,chargeback,23.0
4,355,2026-01-05 12:00:00,2026-01-16 12:00:00,chargeback,11.0
5,356,2026-01-18 07:00:00,2026-01-29 07:00:00,chargeback,11.0
6,357,2026-01-19 11:00:00,2026-01-23 11:00:00,chargeback,4.0
7,358,2026-01-06 05:00:00,2026-01-28 05:00:00,chargeback,22.0
8,359,2026-01-18 22:00:00,2026-01-25 22:00:00,chargeback,7.0
9,360,2025-12-22 13:00:00,2026-01-13 13:00:00,chargeback,22.0


In [5]:
query_4 = """
select 
user_id,
substr(transaction_time, 1, 15) || '0:00' as time_bucket,
count(transaction_id) as transaction_count
from transactions
group by user_id,
substr(transaction_time, 1, 15)
having 
count(transaction_id) >= 3
"""

print("Velocity Attackers (10-Minute Buckets)")
display(pd.read_sql(query_4, conn))

Velocity Attackers (10-Minute Buckets)


,user_id,time_bucket,transaction_count
0,59,2026-01-09 21:00:00,4
1,73,2026-01-12 09:00:00,4
2,154,2026-01-02 22:00:00,4
3,200,2026-01-01 22:00:00,4
4,229,2026-01-12 12:00:00,4
5,287,2026-01-14 14:00:00,4
6,314,2026-01-02 18:00:00,4
7,345,2026-01-23 09:00:00,4


In [6]:
query_5="""
select m.merchant_id, m.merchant_name,
count(distinct t.transaction_id) as valid_txn_volume,
sum(t.amount_inr) as "Net Revenue"
from merchants m
left join transactions t on m.merchant_id=t.merchant_id
where t.status!='chargeback'
and t.user_id not in(
select user_id
from transactions
group by user_id, substr(transaction_time, 1, 15)
having count(transaction_id)>=3
)
group by m.merchant_id, m.merchant_name
order by "Net Revenue" desc
limit 10
"""

print("Top 10 Merchants (valid_txns)")
display(pd.read_sql(query_5, conn))

Top 10 Merchants (valid_txns)


,merchant_id,merchant_name,valid_txn_volume,Net Revenue
0,13,Merchant_013,11,13389
1,7,Merchant_007,13,13137
2,17,Merchant_017,12,12738
3,25,Merchant_025,16,12534
4,22,Merchant_022,15,12135
5,39,Merchant_039,9,11291
6,36,Merchant_036,14,11236
7,37,Merchant_037,18,10932
8,11,Merchant_011,8,10492
9,1,Merchant_001,9,10441


In [7]:
query_6="""
select u.user_id,
sum(t.amount_inr) as "Net Expenses"
from users u
left join transactions t on u.user_id=t.user_id
where t.status='captured'
and t.payment_method='UPI'
and t.user_id not in(
select user_id
from transactions
group by user_id, substr(transaction_time, 1, 15)
having count(transaction_id)>=3
)
group by u.user_id
order by "Net Expenses" desc
limit 10
"""

query_6_1="""
select u.user_id,
sum(t.amount_inr) as "Net Expenses"
from users u
left join transactions t on u.user_id=t.user_id
where t.status='captured'
and t.payment_method='Netbanking'
and t.user_id not in(
select user_id
from transactions
group by user_id, substr(transaction_time, 1, 15)
having count(transaction_id)>=3
)
group by u.user_id
order by "Net Expenses" desc
limit 10
"""

query_6_2="""
select u.user_id,
sum(t.amount_inr) as "Net Expenses"
from users u
left join transactions t on u.user_id=t.user_id
where t.status='captured'
and t.payment_method='Card'
and t.user_id not in(
select user_id
from transactions
group by user_id, substr(transaction_time, 1, 15)
having count(transaction_id)>=3
)
group by u.user_id
order by "Net Expenses" desc
limit 10
"""

query_6_3="""
select u.user_id,
sum(t.amount_inr) as "Net Expenses"
from users u
left join transactions t on u.user_id=t.user_id
where t.status='captured'
and t.payment_method='Wallet'
and t.user_id not in(
select user_id
from transactions
group by user_id, substr(transaction_time, 1, 15)
having count(transaction_id)>=3
)
group by u.user_id
order by "Net Expenses" desc
limit 10
"""

print("Top 10 Spenders_UPI")
display(pd.read_sql(query_6, conn))

print("Top 10 Spenders_Netbanking")
display(pd.read_sql(query_6_1, conn))

print("Top 10 Spenders_Card")
display(pd.read_sql(query_6_2, conn))

print("Top 10 Spenders_Wallet")
display(pd.read_sql(query_6_3, conn))

Top 10 Spenders_UPI


,user_id,Net Expenses
0,186,5498
1,216,5098
2,213,5098
3,151,5048
4,293,4999
5,281,4999
6,239,4999
7,105,4999
8,235,3147
9,170,3098


Top 10 Spenders_Netbanking


,user_id,Net Expenses
0,208,4999
1,38,4999
2,100,2999
3,330,1499
4,325,1499
5,313,1499
6,143,1499
7,128,1499
8,39,948
9,213,799


Top 10 Spenders_Card


,user_id,Net Expenses
0,147,3098
1,65,3098
2,207,2999
3,156,2999
4,42,2999
5,310,1548
6,289,1499
7,277,1499
8,158,1499
9,63,1499


Top 10 Spenders_Wallet


,user_id,Net Expenses
0,149,5298
1,62,4999
2,283,3098
3,252,3098
4,350,2999
5,297,2999
6,145,2999
7,130,2999
8,103,2999
9,60,2999
